In [44]:
# @title
# ===================================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ===================================================================
# NFStream es la librería clave para analizar los archivos .pcap
# El resto son para manipulación de datos, IA y gráficos.
# El -q es para que la salida de la instalación sea más limpia (quiet).
# ===================================================================
!pip install -q \
    nfstream \
    pandas==2.2.3 \
    numpy==2.1.3 \
    scikit-learn==1.6.1 \
    matplotlib==3.10.0 \
    seaborn==0.13.2 \
    joblib==1.5.3 \
    xgboost==3.4.1 \
    tensorflow==2.20.0 \
    psutil==5.9.5


In [45]:
# @title
# ===================================================================
# CELDA 2: IMPORTACIÓN DE MÓDULOS Y CONFIGURACIÓN DE GPU
# ===================================================================
import os
import pandas as pd
import numpy as np
import math
import re
import random
from datetime import datetime
from nfstream import NFStreamer
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import seaborn as sns
import joblib
import xgboost as xgb
import warnings

import time
import psutil
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Conv2D, Flatten, Dropout, MaxPooling1D, MaxPooling2D, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PATH_DRIVE = '/content/drive/MyDrive/NIDS'
PATH_DRIVE_DATA = PATH_DRIVE + '/Partidas'
# Ignoramos advertencias para mantener la salida limpia
warnings.filterwarnings('ignore')

# Verificación segura de GPU antes de que se inicie el runtime de Keras
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[OK] TensorFlow detectó y configuró la GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(f"[OK] GPU detectada (Runtime de TensorFlow ya inicializado).")
else:
    print("[AVISO] No se detectó ninguna GPU activa; TensorFlow funcionará en CPU.")

print("Librerías importadas correctamente.")

SEMILLA_GLOBAL = 42

random.seed(SEMILLA_GLOBAL)
np.random.seed(SEMILLA_GLOBAL)
tf.random.set_seed(SEMILLA_GLOBAL)
os.environ['PYTHONHASHSEED'] = str(SEMILLA_GLOBAL)

import sys
import platform
import sklearn
import hashlib
from datetime import datetime, timezone
import threading

Mounted at /content/drive
[AVISO] No se detectó ninguna GPU activa; TensorFlow funcionará en CPU.
Librerías importadas correctamente.


In [46]:
# @title
# ===================================================================
# CELDA 3: FUNCIÓN INTELIGENTE DE DESFASE
# ===================================================================
def obtener_desfase_por_csv(archivo_sync, archivo_csv):
    """
    Calcula el desfase temporal entre el timestamp del juego y el timestamp UNIX
    buscando una marca de sincronización en un archivo de log.
    """
    print(f"   -> Buscando sincronización para {archivo_csv}...")

    # Extrae la fecha y hora del nombre del fichero de telemetría (ej: YYYYMMDD_HHMMSS)
    match_csv = re.search(r"(\d{8}_\d{6})", archivo_csv)
    if not match_csv:
        raise ValueError("El nombre del CSV no tiene el formato de fecha esperado (YYYYMMDD_HHMMSS).")
    fecha_csv = match_csv.group(1)

    with open(archivo_sync, 'r') as f:
        for linea in f: # Buscar linea por linea en el archivo de sincronización
            patron = r"\[([\d\.]+)\] ==== \[SYNC_TELEMETRIA\] Tiempo interno: ([\d\.]+) ===="
            match = re.search(patron, linea) # Extrae el tiempo UNIX y el tiempo del juego
            if match:
                tiempo_unix = float(match.group(1))
                tiempo_juego_sec = float(match.group(2))

                # Compara si la fecha y hora del log coincide con la del archivo CSV
                fecha_unix_str = datetime.fromtimestamp(tiempo_unix).strftime("%Y%m%d_%H%M%S")
                if fecha_unix_str == fecha_csv:
                    desfase = tiempo_unix - tiempo_juego_sec
                    print(f"      [OK] Sincronización encontrada. Desfase: {desfase:.3f} s")
                    return desfase

    raise ValueError(f"No se encontró en '{archivo_sync}' la fecha correspondiente al CSV '{archivo_csv}'")

def cargar_bloques_tfg_etiquetas(ruta_csv):
    bloques = []
    bloque_actual = []
    ultimo_tiempo_por_jugador = {}

    with open(ruta_csv, 'r') as f:
        for linea in f:
            partes = linea.strip().split(',')
            if len(partes) < 3:
                continue

            tiempo = float(partes[0])
            player_id = int(partes[1])

            tiempo_anterior = ultimo_tiempo_por_jugador.get(player_id)
            if tiempo_anterior is not None and tiempo < tiempo_anterior:
                bloques.append(bloque_actual)
                bloque_actual = []
                ultimo_tiempo_por_jugador = {}

            bloque_actual.append(linea)
            ultimo_tiempo_por_jugador[player_id] = tiempo

    if bloque_actual:
        bloques.append(bloque_actual)

    return bloques


In [ ]:
# @title
# ===================================================================
# CELDA 4: PREPROCESAMIENTO, FUSIÓN Y CARACTERÍSTICAS V3 (CORREGIDA)
# ===================================================================

def preprocesar_y_fusionar(archivo_csv, pcap_local, pcap_amigo, desfase,
                           tipo_red, es_ataque, fase, tipo_ataque,
                           usar_ventanas, usar_v3=False, bloque_aimbot=None):
    """Fusiona telemetría + red y genera características temporales.
    V3 añade ventanas multiescala, variaciones, y picos máximos para Flood.
    """
    df_juego = pd.read_csv(archivo_csv)
    df_juego['timestamp_real'] = df_juego['timestamp'] + desfase
    df_juego = df_juego.sort_values('timestamp_real')
    print(f"   -> Telemetría leída: {len(df_juego)} filas.")

    tiempos_inicio = df_juego.groupby('player_id')['timestamp'].min()

    jugadores_detectados = tiempos_inicio.index.tolist()
    if len(jugadores_detectados) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 jugadores en '{archivo_csv}', "
            f"pero se detectaron {len(jugadores_detectados)}: {jugadores_detectados}. "
            f"Revisa el CSV antes de continuar."
        )

    id_amigo = tiempos_inicio.idxmin()
    id_host = tiempos_inicio.idxmax()
    print(f"      [AUTO-ID] Amigo={id_amigo} | Host={id_host}")

    df_juego_host = df_juego[df_juego['player_id'] == id_host].copy()
    df_juego_amigo = df_juego[df_juego['player_id'] == id_amigo].copy()

    def procesar_pcap(pcap_path, tipo_conexion_str):
        try:
            streamer = NFStreamer(source=pcap_path, statistical_analysis=True, active_timeout=1)
            df_red = streamer.to_pandas()
            if df_red is None or df_red.empty:
                return pd.DataFrame()

            # Agrupar en ventanas de 100ms (0.1s)
            df_red['timestamp_window'] = np.floor((df_red['bidirectional_first_seen_ms'] / 1000.0) * 10) / 10.0

            df_agg = df_red.groupby('timestamp_window').agg({
                'src2dst_packets': ['sum', 'max'],
                'dst2src_packets': ['sum', 'max'],
                'src2dst_bytes': ['sum', 'max'],
                'dst2src_bytes': ['sum', 'max'],
                'bidirectional_bytes': ['sum', 'max'],
                'bidirectional_duration_ms': 'mean',
                'bidirectional_mean_piat_ms': 'mean',
                'bidirectional_stddev_piat_ms': 'max',
                'bidirectional_mean_ps': 'mean',
                'bidirectional_stddev_ps': 'mean'
            })

            # Aplanamiento seguro para evitar MultiIndex
            df_agg.columns = ['_'.join(col).strip('_') if col[1] != '' else col[0] for col in df_agg.columns.values]
            df_agg = df_agg.reset_index()

            # Las columnas con una sola función de agregación quedan con un sufijo
            # redundante (p. ej. '_mean'); las renombramos a su nombre base esperado.
            df_agg = df_agg.rename(columns={
                'bidirectional_duration_ms_mean': 'bidirectional_duration_ms',
                'bidirectional_mean_piat_ms_mean': 'bidirectional_mean_piat_ms',
                'bidirectional_stddev_piat_ms_max': 'bidirectional_stddev_piat_ms',
                'bidirectional_mean_ps_mean': 'bidirectional_mean_ps',
                'bidirectional_stddev_ps_mean': 'bidirectional_stddev_ps',
            })

            df_agg['tipo_conexion'] = tipo_conexion_str
            return df_agg.sort_values('timestamp_window')
        except Exception as e:
            print(f"      [AVISO] Error procesando PCAP {pcap_path}: {e}")
            return pd.DataFrame()

    # --- BLOQUE DE FUSIÓN ---
    df_red_local = procesar_pcap(pcap_local, 'local')
    df_combinado_host = pd.DataFrame()
    if not df_red_local.empty and not df_juego_host.empty:
        df_combinado_host = pd.merge_asof(
            df_juego_host.sort_values('timestamp_real'),
            df_red_local.sort_values('timestamp_window'),
            left_on='timestamp_real', right_on='timestamp_window',
            direction='backward', tolerance=0.5
        )

    df_red_amigo = procesar_pcap(pcap_amigo, 'amigo')
    if df_red_amigo.empty or df_juego_amigo.empty:
        raise RuntimeError(f"No se pudieron obtener datos de red para el amigo: {archivo_csv}")

    df_combinado_amigo = pd.merge_asof(
        df_juego_amigo.sort_values('timestamp_real'),
        df_red_amigo.sort_values('timestamp_window'),
        left_on='timestamp_real', right_on='timestamp_window',
        direction='backward', tolerance=0.5
    )

    df_combinado = pd.concat([df_combinado_host, df_combinado_amigo], ignore_index=True)
    df_combinado['red_match'] = df_combinado['timestamp_window'].notna()

    porcentaje_sin_match = (1 - df_combinado['red_match'].mean()) * 100
    print(f"      [RED] Filas sin correspondencia directa en el PCAP: {porcentaje_sin_match:.2f}%")

    # --- RELLENO Y LIMPIEZA DE RED ROBUSTA ---
    # Listado adaptado a las nuevas columnas con sufijos _sum y _max
    columnas_red = [
        'src2dst_packets_sum', 'src2dst_packets_max',
        'dst2src_packets_sum', 'dst2src_packets_max',
        'src2dst_bytes_sum', 'src2dst_bytes_max',
        'dst2src_bytes_sum', 'dst2src_bytes_max',
        'bidirectional_bytes_sum', 'bidirectional_bytes_max',
        'bidirectional_duration_ms', 'bidirectional_mean_piat_ms',
        'bidirectional_stddev_piat_ms', 'bidirectional_mean_ps', 'bidirectional_stddev_ps'
    ]

    # Tiempo transcurrido desde el último paquete de red real (antes de cualquier ffill)
    df_combinado['tiempo_desde_ultimo_paquete'] = np.nan
    mascara_match = df_combinado['red_match']
    df_combinado.loc[mascara_match, 'tiempo_desde_ultimo_paquete'] = df_combinado.loc[mascara_match, 'timestamp_real']
    df_combinado['tiempo_desde_ultimo_paquete'] = (
        df_combinado.groupby('player_id')['tiempo_desde_ultimo_paquete']
        .ffill()
    )
    df_combinado['tiempo_desde_ultimo_paquete'] = (
        df_combinado['timestamp_real'] - df_combinado['tiempo_desde_ultimo_paquete']
    ).fillna(0).clip(lower=0)

    columnas_existentes = [c for c in columnas_red if c in df_combinado.columns]
    if columnas_existentes:
        df_combinado[columnas_existentes] = df_combinado.groupby('player_id')[columnas_existentes].ffill(limit=3)
        df_combinado[columnas_existentes] = df_combinado[columnas_existentes].fillna(0)

    df_combinado = df_combinado.sort_values(['player_id', 'timestamp_real']).copy()

    columnas_numericas = df_combinado.select_dtypes(include='number').columns
    columnas_rellenables = [
        columna for columna in columnas_numericas
        if columna not in ['player_id', 'timestamp_real']
    ]

    df_combinado[columnas_rellenables] = (
        df_combinado
        .groupby('player_id', sort=False)[columnas_rellenables]
        .ffill()
        .fillna(0)
    )
    df_combinado['sesion'] = os.path.basename(archivo_csv)

    # Referencia temporal por jugador, usada para el recorte final de la partida
    df_combinado['tiempo_desde_inicio'] = df_combinado['timestamp_real'] - df_combinado.groupby('player_id')['timestamp_real'].transform('min')

    # CARACTERÍSTICAS BASE
    g = df_combinado.groupby('player_id', group_keys=False)
    diff_yaw = g['yaw'].diff().fillna(0)
    diff_pitch = g['pitch'].diff().fillna(0)
    df_combinado['delta_yaw'] = ((diff_yaw + 180) % 360 - 180).abs()
    df_combinado['delta_pitch'] = ((diff_pitch + 180) % 360 - 180).abs()
    df_combinado['delta_yaw_en_disparo'] = df_combinado['delta_yaw'] * df_combinado['is_attacking']
    df_combinado['delta_pitch_en_disparo'] = df_combinado['delta_pitch'] * df_combinado['is_attacking']

    # Usar columnas sumadas para ratio general
    bytes_base = 'bidirectional_bytes_sum' if 'bidirectional_bytes_sum' in df_combinado.columns else 'bidirectional_bytes'
    df_combinado['ratio_velocidad_bytes'] = df_combinado['velocity'] / (df_combinado[bytes_base] + 1)

    if usar_ventanas:
        df_combinado['delta_t'] = g['timestamp_real'].diff()
        df_combinado.loc[df_combinado['delta_t'] <= 0, 'delta_t'] = np.nan
        df_combinado['delta_t'] = (
            df_combinado.groupby('player_id')['delta_t']
            .transform(lambda serie: serie.ffill())
        )

        df_combinado['vel_angular_yaw'] = (
            df_combinado['delta_yaw'] / df_combinado['delta_t']
        ).fillna(0)

        df_combinado['vel_angular_pitch'] = (
            df_combinado['delta_pitch'] / df_combinado['delta_t']
        ).fillna(0)

        df_combinado['acc_angular_yaw'] = (
            g['vel_angular_yaw'].diff() / df_combinado['delta_t']
        ).fillna(0)

        df_combinado['acc_angular_pitch'] = (
            g['vel_angular_pitch'].diff() / df_combinado['delta_t']
        ).fillna(0)

        df_combinado['jerk_angular_yaw'] = (
            g['acc_angular_yaw'].diff() / df_combinado['delta_t']
        ).fillna(0)

        df_combinado['jerk_angular_pitch'] = (
            g['acc_angular_pitch'].diff() / df_combinado['delta_t']
        ).fillna(0)

        columnas_ventana = [
          'velocity', 'vel_angular_yaw', 'vel_angular_pitch',
          'acc_angular_yaw', 'acc_angular_pitch',
          'src2dst_packets_sum', 'dst2src_packets_sum',
          'src2dst_bytes_sum', 'dst2src_bytes_sum',
          'bidirectional_bytes_sum',
          'bidirectional_mean_piat_ms',
          'bidirectional_stddev_piat_ms'
        ]
        columnas_ventana = [c for c in columnas_ventana if c in df_combinado.columns]

        ventanas = [5, 10, 20, 50, 100] if usar_v3 else [30]

        for w in ventanas:
            df_combinado[f'red_match_roll_mean_{w}'] = (
                df_combinado.groupby('player_id')['red_match']
                .transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).mean())
                .fillna(1.0).astype('float32')
            )

        for col in columnas_ventana:
            serie = df_combinado.groupby('player_id')[col]
            for w in ventanas:
                # Convertimos a float32 al vuelo para ahorrar RAM
                df_combinado[f'{col}_roll_mean_{w}'] = serie.transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).mean()).fillna(0).astype('float32')
                df_combinado[f'{col}_roll_std_{w}'] = serie.transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).std()).fillna(0).astype('float32')

                if usar_v3:
                    df_combinado[f'{col}_roll_max_{w}'] = serie.transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).max()).fillna(0).astype('float32')
                    df_combinado[f'{col}_roll_min_{w}'] = serie.transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).min()).fillna(0).astype('float32')
                    # df_combinado[f'{col}_roll_q95_{w}'] = serie.transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).quantile(0.95)).fillna(0).astype('float32')

        if usar_v3:
            cols_diff = [c for c in ['src2dst_packets_sum', 'dst2src_packets_sum', 'src2dst_bytes_sum', 'dst2src_bytes_sum', 'bidirectional_bytes_sum'] if c in df_combinado.columns]
            for col in cols_diff:
                df_combinado[f'{col}_diff'] = g[col].diff().fillna(0).astype('float32')
                df_combinado[f'{col}_pct_change'] = g[col].pct_change().replace([np.inf, -np.inf], 0).fillna(0).clip(-10, 10).astype('float32')

            for w in [5, 10, 20, 50]:
                for col in ['src2dst_packets_sum', 'dst2src_packets_sum', 'bidirectional_bytes_sum']:
                    if col not in df_combinado.columns: continue
                    mean_temp = g[col].transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).mean())
                    std_temp = g[col].transform(lambda x, ww=w: x.rolling(ww, min_periods=max(2, ww // 3)).std()).fillna(0)
                    df_combinado[f'{col}_cv_{w}'] = (std_temp / (mean_temp.abs() + 1e-6)).replace([np.inf, -np.inf], 0).fillna(0).astype('float32')

                    # Destruimos los objetos temporales pesados de Pandas
                    del mean_temp
                    del std_temp
                    gc.collect()

            # Características globales para Flood y Ratios (¡Las que faltaban!)
            p_src = 'src2dst_packets_sum' if 'src2dst_packets_sum' in df_combinado.columns else 'src2dst_packets'
            p_dst = 'dst2src_packets_sum' if 'dst2src_packets_sum' in df_combinado.columns else 'dst2src_packets'
            b_src = 'src2dst_bytes_sum' if 'src2dst_bytes_sum' in df_combinado.columns else 'src2dst_bytes'
            b_dst = 'dst2src_bytes_sum' if 'dst2src_bytes_sum' in df_combinado.columns else 'dst2src_bytes'

            df_combinado['packets_total'] = df_combinado[p_src] + df_combinado[p_dst]
            df_combinado['bytes_total'] = df_combinado[b_src] + df_combinado[b_dst]

            df_combinado['packets_per_second'] = (df_combinado['packets_total'] / df_combinado['delta_t'].clip(lower=0.001)).fillna(0)
            df_combinado['bytes_per_second'] = (df_combinado['bytes_total'] / df_combinado['delta_t'].clip(lower=0.001)).fillna(0)

            df_combinado['packets_total_diff'] = g['packets_total'].diff().fillna(0)
            df_combinado['bytes_total_diff'] = g['bytes_total'].diff().fillna(0)
            df_combinado['packets_acceleration'] = g['packets_total_diff'].diff().fillna(0)
            df_combinado['bytes_acceleration'] = g['bytes_total_diff'].diff().fillna(0)

            # Ratios y velocidades totales recuperadas
            df_combinado['bytes_ratio'] = (df_combinado[b_src] / (df_combinado[b_dst] + 1)).astype('float32')
            df_combinado['packets_ratio'] = (df_combinado[p_src] / (df_combinado[p_dst] + 1)).astype('float32')
            df_combinado['angular_speed_total'] = np.sqrt(df_combinado['vel_angular_yaw']**2 + df_combinado['vel_angular_pitch']**2).astype('float32')
            df_combinado['angular_acc_total'] = np.sqrt(df_combinado['acc_angular_yaw']**2 + df_combinado['acc_angular_pitch']**2).astype('float32')

    # --- ETIQUETADO ---
    df_combinado['tipo_red'] = tipo_red
    df_combinado['fase'] = fase
    df_combinado['tipo_ataque'] = tipo_ataque if es_ataque else 'ninguno'
    df_combinado['etiqueta_real'] = 1

    if es_ataque:
      intervalos_ataque = []

      def procesar_lineas_evento(lineas):
          inicios_por_jugador = {}
          inicio_temporal_general = None
          resultado = []
          for linea in lineas:
              partes = linea.strip().replace(',', ' ').split()
              if len(partes) < 2:
                  continue
              try:
                  tiempo_leido = float(partes[0])
                  accion = next((p.upper() for p in partes if p.upper() in ['START', 'STOP']), None)
                  if not accion:
                      continue
                  player_id_leido = int(partes[1]) if len(partes) >= 3 and partes[1].isdigit() else None
                  tiempo_unix_real = tiempo_leido + desfase if tiempo_leido < 1e7 else tiempo_leido

                  if accion == 'START':
                      if player_id_leido is not None:
                          inicios_por_jugador[player_id_leido] = tiempo_unix_real
                      else:
                          inicio_temporal_general = tiempo_unix_real
                  elif accion == 'STOP':
                      if player_id_leido is not None and player_id_leido in inicios_por_jugador:
                          resultado.append([inicios_por_jugador[player_id_leido], tiempo_unix_real, player_id_leido])
                          del inicios_por_jugador[player_id_leido]
                      elif player_id_leido is None and inicio_temporal_general is not None:
                          resultado.append([inicio_temporal_general, tiempo_unix_real, None])
                          inicio_temporal_general = None
              except (ValueError, TypeError):
                  continue
          return resultado

      if tipo_ataque in ['Flood', 'LagSwitch']:
          ruta_registro = f"{PATH_DRIVE_DATA}/registro_trampas.txt"
          if os.path.exists(ruta_registro):
              with open(ruta_registro, 'r') as f:
                  intervalos_ataque = procesar_lineas_evento(f.readlines())

      elif tipo_ataque == 'Aimbot':
          ruta_registro = f"{PATH_DRIVE_DATA}/tfg_etiquetas.csv"
          if not os.path.exists(ruta_registro):
              raise FileNotFoundError(f"No se encuentra {ruta_registro}")
          if bloque_aimbot is None:
              raise ValueError("Debes indicar 'bloque_aimbot' para sesiones Aimbot.")

          bloques = cargar_bloques_tfg_etiquetas(ruta_registro)
          if bloque_aimbot >= len(bloques):
              raise ValueError(
                  f"bloque_aimbot={bloque_aimbot} fuera de rango; "
                  f"solo hay {len(bloques)} bloques en tfg_etiquetas.csv"
              )

          # Solo nos interesa el id que ya fue detectado como amigo para esta sesión
          lineas_amigo = [
              linea for linea in bloques[bloque_aimbot]
              if int(linea.strip().split(',')[1]) == id_amigo
          ]
          intervalos_ataque = procesar_lineas_evento(lineas_amigo)

      if intervalos_ataque:
        tiempo_min = df_combinado['timestamp_real'].min()
        tiempo_max = df_combinado['timestamp_real'].max()
        for inicio, fin, id_tramposo in intervalos_ataque:
            if inicio <= tiempo_max and fin >= tiempo_min:
                mascara = (df_combinado['timestamp_real'] >= inicio) & (df_combinado['timestamp_real'] <= fin)
                if id_tramposo is not None:
                    mascara &= df_combinado['player_id'] == id_tramposo
                df_combinado.loc[mascara, 'etiqueta_real'] = -1
                df_combinado.loc[mascara, 'tipo_ataque'] = tipo_ataque

    tiempo_maximo_por_jugador = (df_combinado.groupby('player_id')['timestamp_real'].transform('max'))

    df_combinado = df_combinado[df_combinado['timestamp_real'] <= tiempo_maximo_por_jugador - 5.0].copy()
    df_combinado.loc[df_combinado['etiqueta_real'] == 1, 'tipo_ataque'] = 'ninguno'

    return df_combinado

In [48]:
# @title
# ===================================================================
# CELDA 5: ENTRENADOR UNIVERSAL Y ESTUDIO DE ABLACIÓN (CON MÉTRICAS)
# ===================================================================

# Configuración base de detección
IF_CONTAMINATION = 0.005
CONF_THRESHOLDS = {'LagSwitch': 0.60, 'Flood': 0.60, 'Aimbot': 0.60}

class MonitorRecursos:
    """Muestrea RAM y CPU en segundo plano para capturar el pico real, no solo antes/después."""
    def __init__(self, intervalo=0.2):
        self.intervalo = intervalo
        self.proceso = psutil.Process(os.getpid())
        self.pico_ram_mb = 0.0
        self.pico_cpu_pct = 0.0
        self._detener = threading.Event()
        self._hilo = None

    def _muestrear(self):
        while not self._detener.is_set():
            ram_actual = self.proceso.memory_info().rss / (1024 * 1024)
            cpu_actual = self.proceso.cpu_percent(interval=None)
            self.pico_ram_mb = max(self.pico_ram_mb, ram_actual)
            self.pico_cpu_pct = max(self.pico_cpu_pct, cpu_actual)
            self._detener.wait(self.intervalo)

    def __enter__(self):
        self.proceso.cpu_percent(interval=None)  # descarta la primera lectura (siempre 0.0)
        self._detener.clear()
        self._hilo = threading.Thread(target=self._muestrear, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self._detener.set()
        self._hilo.join()

def asignar_validacion_por_eventos(df, fases_entrenables=('train', 'train_rf'),
                                    fraccion_eventos_validacion=0.2,
                                    margen_segundos=5.0):
    """Marca como validación una fracción de los eventos de ataque de cada sesión,
    incluyendo un margen de contexto normal alrededor. El resto queda para entrenamiento.
    """
    df = df.copy()
    df['subfase'] = 'entrenamiento'

    mascara_entrenable = (
        df['tipo_conexion'].eq('amigo') &
        df['fase'].isin(fases_entrenables)
    )

    for sesion in df.loc[mascara_entrenable, 'sesion'].unique():
        mascara_sesion = mascara_entrenable & (df['sesion'] == sesion)
        sub = df.loc[mascara_sesion].sort_values('timestamp_real')

        es_ataque = sub['etiqueta_real'] == -1
        if not es_ataque.any():
            continue

        cambio_estado = es_ataque != es_ataque.shift()
        id_evento_local = cambio_estado.cumsum()

        eventos = sub[es_ataque].groupby(id_evento_local)
        ids_eventos = list(eventos.groups.keys())

        n_eventos = len(ids_eventos)
        n_validacion = max(1, round(n_eventos * fraccion_eventos_validacion))
        ids_validacion = ids_eventos[-n_validacion:]

        for id_evento in ids_validacion:
            bloque = eventos.get_group(id_evento)
            inicio = bloque['timestamp_real'].min() - margen_segundos
            fin = bloque['timestamp_real'].max() + margen_segundos

            mascara_ventana = (
                mascara_sesion &
                (df['timestamp_real'] >= inicio) &
                (df['timestamp_real'] <= fin)
            )
            df.loc[mascara_ventana, 'subfase'] = 'validacion'

    return df

def _crear_features(df, usar_ventanas, usar_v3):
    base = [
        'velocity', 'delta_yaw', 'delta_pitch', 'is_attacking',
        'delta_yaw_en_disparo', 'delta_pitch_en_disparo',
        'tiempo_desde_ultimo_paquete',
        'src2dst_packets_sum', 'dst2src_packets_sum', 'src2dst_bytes_sum', 'dst2src_bytes_sum',
        'bidirectional_bytes_sum', 'bidirectional_duration_ms', 'bidirectional_mean_piat_ms',
        'bidirectional_stddev_piat_ms', 'bidirectional_mean_ps', 'bidirectional_stddev_ps',
        'ratio_velocidad_bytes'
    ]
    if usar_ventanas:
        base += ['vel_angular_yaw', 'vel_angular_pitch', 'acc_angular_yaw', 'acc_angular_pitch',
                 'jerk_angular_yaw', 'jerk_angular_pitch']
        ventanas = [5, 10, 20, 50, 100] if usar_v3 else [30]
        base += [f'red_match_roll_mean_{w}' for w in ventanas]
        cols = ['velocity', 'vel_angular_yaw', 'vel_angular_pitch', 'acc_angular_yaw', 'acc_angular_pitch',
                'src2dst_packets_sum', 'dst2src_packets_sum', 'src2dst_bytes_sum', 'dst2src_bytes_sum',
                'bidirectional_bytes_sum', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms']
        for col in cols:
            for w in ventanas:
                base += [f'{col}_roll_mean_{w}', f'{col}_roll_std_{w}']
                if usar_v3:
                    base += [f'{col}_roll_max_{w}', f'{col}_roll_min_{w}']
        if usar_v3:
            base += [
                'bytes_ratio', 'packets_ratio', 'angular_speed_total', 'angular_acc_total',
                'src2dst_packets_sum_diff', 'dst2src_packets_sum_diff', 'src2dst_bytes_sum_diff',
                'dst2src_bytes_sum_diff', 'bidirectional_bytes_sum_diff',
                'src2dst_packets_sum_pct_change', 'dst2src_packets_sum_pct_change',
                'src2dst_bytes_sum_pct_change', 'dst2src_bytes_sum_pct_change',
                'bidirectional_bytes_sum_pct_change',
                'packets_total', 'bytes_total', 'packets_per_second', 'bytes_per_second',
                'packets_total_diff', 'bytes_total_diff', 'packets_acceleration', 'bytes_acceleration'
            ]
            for w in [5, 10, 20, 50]:
                base += [f'src2dst_packets_sum_cv_{w}', f'dst2src_packets_sum_cv_{w}', f'bidirectional_bytes_sum_cv_{w}']
    return list(dict.fromkeys(base))

def optimizar_umbrales_por_clase(probs_val, clases, y_real_val, rango_umbrales=np.arange(0.3, 0.91, 0.05)):
    """Busca, para cada clase de ataque, el umbral que maximiza el F1 en validación."""
    from sklearn.metrics import f1_score

    indices = np.argmax(probs_val, axis=1)
    max_probs = np.max(probs_val, axis=1)
    predicciones_base = clases[indices]

    umbrales_optimos = {}
    for clase in clases:
        if clase == 'ninguno':
            continue
        mejor_umbral, mejor_f1 = 0.5, -1
        y_real_binaria = (y_real_val == clase).astype(int)
        for umbral in rango_umbrales:
            pred_binaria = ((predicciones_base == clase) & (max_probs >= umbral)).astype(int)
            f1 = f1_score(y_real_binaria, pred_binaria, zero_division=0)
            if f1 > mejor_f1:
                mejor_f1, mejor_umbral = f1, umbral
        umbrales_optimos[clase] = float(mejor_umbral)

    return umbrales_optimos

def _aplicar_umbral_por_clase(probs, clases, predicciones_if=None, umbrales=None):
    if umbrales is None:
        umbrales = CONF_THRESHOLDS
    indices = np.argmax(probs, axis=1)
    max_probs = np.max(probs, axis=1)
    predicciones = clases[indices]
    if predicciones_if is None:
        predicciones_if = np.ones(len(predicciones))
    resultado = []
    for pred, prob, pred_if in zip(predicciones, max_probs, predicciones_if):
        if pred != 'ninguno' and prob >= umbrales.get(str(pred), 0.50):
            resultado.append(pred)
        else:
            if pred_if == -1:
                resultado.append('Anomalia_Desconocida')
            else:
                resultado.append('ninguno')
    return np.array(resultado, dtype=object), max_probs

def filtro_histeresis(df, pred_col='prediccion_ataque', n_consecutivos=3, tolerancia=1):
    """Como el original, pero permite hasta 'tolerancia' predicciones discordantes
    dentro de la racha antes de reiniciar el contador del candidato."""
    df = df.copy()
    out = np.full(len(df), 'ninguno', dtype=object)
    for _, idx in df.groupby(['sesion', 'player_id'], sort=False).groups.items():
        idx = np.array(list(idx))
        sub = df.loc[idx].sort_values('timestamp_real')
        vals = sub[pred_col].tolist()
        pred_limpias, estado_actual, candidato, aciertos, fallos = [], 'ninguno', None, 0, 0
        for p in vals:
            if p == estado_actual:
                candidato, aciertos, fallos = None, 0, 0
                pred_limpias.append(estado_actual)
                continue
            if p == candidato:
                aciertos += 1
            elif candidato is None:
                candidato, aciertos, fallos = p, 1, 0
            else:
                fallos += 1
                if fallos > tolerancia:
                    candidato, aciertos, fallos = p, 1, 0
            if aciertos >= n_consecutivos:
                estado_actual, candidato, aciertos, fallos = candidato, None, 0, 0
            pred_limpias.append(estado_actual)
        out[sub.index.to_numpy()] = pred_limpias
    df['prediccion_ataque_suavizada'] = out
    return df

class _TeeStdout:
    """Duplica la salida de print() hacia la consola y un buffer en memoria."""
    def __init__(self, *destinos):
        self.destinos = destinos
    def write(self, mensaje):
        for destino in self.destinos:
            destino.write(mensaje)
    def flush(self):
        for destino in self.destinos:
            destino.flush()

def generar_informe_calidad_dataset(df, ruta_salida=None):
    """Resumen de calidad para detectar problemas antes de entrenar."""
    import io
    buffer = io.StringIO()
    stdout_original = sys.stdout
    sys.stdout = _TeeStdout(stdout_original, buffer)

    print("\n" + "=" * 65)
    print("INFORME DE CALIDAD DEL DATASET")
    print("=" * 65)

    df_amigo = df[df['tipo_conexion'] == 'amigo']

    print(f"Filas totales (amigo):        {len(df_amigo)}")
    print(f"Sesiones distintas (amigo):    {df_amigo['sesion'].nunique()}")

    print("\n--- Filas por fase y tipo de ataque ---")
    resumen_fase = (
        df_amigo
        .groupby(['fase', 'tipo_ataque'])
        .size()
        .rename('filas')
        .reset_index()
    )
    print(resumen_fase.to_string(index=False))

    print("\n--- Balance de clases (etiqueta_real) por fase ---")
    for fase in sorted(df_amigo['fase'].unique()):
        sub = df_amigo[df_amigo['fase'] == fase]
        buenas = (sub['etiqueta_real'] == 1).sum()
        malas = (sub['etiqueta_real'] == -1).sum()
        total = len(sub)
        porcentaje_malas = (malas / total * 100) if total else 0
        print(f"  {fase:<10} -> normales: {buenas:>6} | ataques: {malas:>6} ({porcentaje_malas:.2f}%)")

    print("\n--- Eventos de ataque por sesión ---")
    for sesion in df_amigo['sesion'].unique():
        sub = df_amigo[df_amigo['sesion'] == sesion].sort_values('timestamp_real')
        es_ataque = sub['etiqueta_real'] == -1
        if not es_ataque.any():
            print(f"  {sesion:<45} -> 0 eventos (sesión normal)")
            continue
        cambio = es_ataque != es_ataque.shift()
        num_eventos = len(sub[es_ataque].groupby(cambio.cumsum()))
        print(f"  {sesion:<45} -> {num_eventos} eventos")

    print("\n--- Valores nulos ---")
    nulos = df_amigo.isnull().sum()
    nulos = nulos[nulos > 0]
    if nulos.empty:
        print("  Sin valores nulos detectados.")
    else:
        for columna, cantidad in nulos.items():
            print(f"  {columna:<40} -> {cantidad} nulos ({cantidad / len(df_amigo) * 100:.2f}%)")

    print("\n--- Comprobación de solapamiento temporal entre fases ---")
    rangos = (
        df_amigo
        .groupby(['sesion', 'fase'])['timestamp_real']
        .agg(['min', 'max'])
        .reset_index()
    )
    fases_por_sesion = df_amigo.groupby('sesion')['fase'].nunique()
    sesiones_multi_fase = fases_por_sesion[fases_por_sesion > 1]
    if sesiones_multi_fase.empty:
        print("  Cada sesión pertenece a una única fase (correcto).")
    else:
        print("  [AVISO] Sesiones que aparecen en más de una fase:")
        print(f"  {list(sesiones_multi_fase.index)}")

    print("\n--- Cobertura de red (PCAP) ---")
    for sesion in df_amigo['sesion'].unique():
        sub = df_amigo[df_amigo['sesion'] == sesion]
        if 'red_match' not in sub.columns:
            continue
        cobertura = sub['red_match'].mean() * 100
        print(f"  {sesion:<45} -> {cobertura:.2f}% de filas con match directo en PCAP")

    if 'delta_t' in df_amigo.columns:
        print("\n--- Regularidad del muestreo temporal (delta_t) ---")
        for sesion in df_amigo['sesion'].unique():
            sub = df_amigo[(df_amigo['sesion'] == sesion) & (df_amigo['delta_t'] > 0)]
            if sub.empty:
                continue
            mediana = sub['delta_t'].median()
            desviacion = sub['delta_t'].std()
            cv = (desviacion / mediana * 100) if mediana else 0
            aviso = "  [AVISO] muestreo irregular" if cv > 30 else ""
            print(
                f"  {sesion:<45} -> delta_t mediana: {mediana*1000:.1f} ms | "
                f"CV: {cv:.1f}%{aviso}"
            )
    print("=" * 65 + "\n")

    sys.stdout = stdout_original
    if ruta_salida:
        with open(ruta_salida, 'w', encoding='utf-8') as f:
            f.write(buffer.getvalue())
        print(f"[OK] Informe de calidad guardado en: {ruta_salida}")

def entrenar_nids(df_completo, usar_ventanas, usar_v3=False, tipo_modelo='RF', usar_if_consejero=True):
    print(f"\n[AI] Preparando entrenamiento: {tipo_modelo} | ¿Usa IF Consejero?: {usar_if_consejero}")

    # ESTADÍSTICAS DEL DATASET
    filas_totales = len(df_completo)
    filas_buenas = len(df_completo[df_completo['etiqueta_real'] == 1])
    filas_malas = len(df_completo[df_completo['etiqueta_real'] == -1])

    print(f"   -> Filas procesadas totales: {filas_totales}")
    print(f"   -> Filas buenas (Normales):  {filas_buenas} ({filas_buenas/filas_totales*100:.2f}%)")
    print(f"   -> Filas malas (Ataques):    {filas_malas} ({filas_malas/filas_totales*100:.2f}%)")

    features = _crear_features(df_completo, usar_ventanas, usar_v3)
    features_faltantes = [
      feature for feature in features
      if feature not in df_completo.columns
    ]

    if features_faltantes:
        raise ValueError(
            "Faltan features requeridas en el dataset: "
            + ", ".join(features_faltantes)
        )

    df_completo[features] = (
      df_completo[features]
      .replace([np.inf, -np.inf], 0)
      .fillna(0)
    )

    scaler = StandardScaler()
    idx_scaler_train = df_completo[
        (df_completo['tipo_conexion'] == 'amigo') &
        (df_completo['fase'] == 'train') &
        (df_completo['etiqueta_real'] == 1) &
        (df_completo['subfase'] == 'entrenamiento')
    ].index

    if len(idx_scaler_train) == 0:
        raise ValueError(
            "No hay muestras normales de entrenamiento para ajustar el scaler."
    )

    scaler.fit(df_completo.loc[idx_scaler_train, features])
    df_completo_scaled = pd.DataFrame(scaler.transform(df_completo[features]), columns=features, index=df_completo.index)

    # Variables de medición de recursos de entrenamiento
    monitor_train = MonitorRecursos(intervalo=0.2)
    monitor_train.__enter__()
    tiempo_inicio_train = time.time()

    modelo_if = None
    # 1. Isolation Forest (Solo se entrena si se solicita explícitamente como consejero)
    if usar_if_consejero:
        # ¡AQUÍ INYECTAMOS EL CONTEXTO TEMPORAL AL CONSEJERO!
        candidatas_if = [
            # Variables instantáneas
            'bidirectional_stddev_piat_ms',
            'src2dst_packets_sum',
            'bidirectional_bytes_sum',
            'delta_yaw',
            'vel_angular_yaw',

            # Variables de Memoria Temporal (Contexto V3)
            'src2dst_packets_sum_roll_max_5',          # Detecta ráfagas masivas (Flood)
            'bidirectional_bytes_sum_roll_max_5',      # Detecta volumen sostenido (Flood/Lag)
            'bidirectional_mean_piat_ms_roll_std_5'    # Detecta inestabilidad en el ritmo (LagSwitch)
        ]

        # Filtro de seguridad: Solo usa las que existan en el dataset actual
        features_para_if = [f for f in candidatas_if if f in features]
        idx_if_train = df_completo[(df_completo['tipo_conexion'] == 'amigo') & (df_completo['etiqueta_real'] == 1) & (df_completo['fase'] == 'train') & (df_completo['subfase'] == 'entrenamiento')].index

        MINIMO_FILAS_IF = 200
        if len(idx_if_train) < MINIMO_FILAS_IF:
            raise ValueError(
                f"Solo hay {len(idx_if_train)} filas normales de train para el Isolation Forest "
                f"(mínimo recomendado: {MINIMO_FILAS_IF}). Revisa tus sesiones 'train'."
            )

        modelo_if = IsolationForest(n_estimators=300, contamination=IF_CONTAMINATION, random_state=42, n_jobs=-1)
        idx_val_if = df_completo[
            (df_completo['tipo_conexion'] == 'amigo') &
            (df_completo['fase'].isin(['train', 'train_rf'])) &
            (df_completo['subfase'] == 'validacion')
        ].index

        if len(idx_val_if) > 0:
            tasa_ataques_val = (df_completo.loc[idx_val_if, 'etiqueta_real'] == -1).mean()
            contaminacion_dinamica = float(np.clip(tasa_ataques_val, 0.001, 0.20))
        else:
            contaminacion_dinamica = IF_CONTAMINATION

        print(f"   -> Contaminación IF calculada dinámicamente desde validación: {contaminacion_dinamica:.4f}")

        modelo_if = IsolationForest(n_estimators=300, contamination=contaminacion_dinamica, random_state=42, n_jobs=-1)
        modelo_if.fit(df_completo_scaled.loc[idx_if_train, features_para_if])
        df_completo['prediccion_ia'] = modelo_if.predict(df_completo_scaled[features_para_if])

    # 2. Modelo Principal
    idx_train = df_completo[
        (df_completo['tipo_conexion'] == 'amigo') &
        (df_completo['fase'].isin(['train', 'train_rf'])) &
        (df_completo['subfase'] == 'entrenamiento')
    ].index

    idx_val = df_completo[
        (df_completo['tipo_conexion'] == 'amigo') &
        (df_completo['fase'].isin(['train', 'train_rf'])) &
        (df_completo['subfase'] == 'validacion')
    ].index
    X_train_base = df_completo_scaled.loc[idx_train, features].values

    if usar_if_consejero:
        X_train_final = np.column_stack((X_train_base, df_completo.loc[idx_train, 'prediccion_ia'].values))
    else:
        X_train_final = X_train_base

    y_train_texto = df_completo.loc[idx_train, 'tipo_ataque'].values
    le = LabelEncoder()
    y_train_num = le.fit_transform(y_train_texto)

    pesos_clases = compute_sample_weight(class_weight='balanced', y=y_train_num)
    pesos_dict = dict(zip(np.unique(y_train_num), compute_sample_weight('balanced', np.unique(y_train_num))))

    D = 0
    pad_size = 0

    if tipo_modelo == 'XGB':
        print(f"[AI] Entrenando XGBoost...")

        X_val_base = df_completo_scaled.loc[idx_val, features].values
        if usar_if_consejero:
            X_val_final = np.column_stack((X_val_base, df_completo.loc[idx_val, 'prediccion_ia'].values))
        else:
            X_val_final = X_val_base
        y_val_num = le.transform(df_completo.loc[idx_val, 'tipo_ataque'].values)

        modelo = xgb.XGBClassifier(
            objective='multi:softprob', eval_metric='mlogloss',
            n_estimators=250, learning_rate=0.08, max_depth=7,
            n_jobs=-1, random_state=42,
            early_stopping_rounds=20
        )
        modelo.fit(
            X_train_final, y_train_num, sample_weight=pesos_clases,
            eval_set=[(X_val_final, y_val_num)], verbose=False
        )

    elif tipo_modelo == 'ANN':
        print(f"[AI] Entrenando ANN...")
        y_train_cat = to_categorical(y_train_num, num_classes=len(le.classes_))

        X_val_base = df_completo_scaled.loc[idx_val, features].values
        if usar_if_consejero:
            X_val_final = np.column_stack((X_val_base, df_completo.loc[idx_val, 'prediccion_ia'].values))
        else:
            X_val_final = X_val_base

        y_val_num = le.transform(df_completo.loc[idx_val, 'tipo_ataque'].values)
        y_val_cat = to_categorical(y_val_num, num_classes=len(le.classes_))

        modelo = Sequential([
            Input(shape=(X_train_final.shape[1],)),
            Dense(128, activation='relu'), BatchNormalization(), Dropout(0.3),
            Dense(64, activation='relu'), BatchNormalization(),
            Dense(len(le.classes_), activation='softmax')
        ])
        modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        modelo.fit(
            X_train_final, y_train_cat, epochs=50, batch_size=256, verbose=0,
            validation_data=(X_val_final, y_val_cat),
            class_weight=pesos_dict, callbacks=[early_stopping]
        )

    elif tipo_modelo == 'CNN':
        print(f"[AI] Entrenando CNN 1D...")
        y_train_cat = to_categorical(y_train_num, num_classes=len(le.classes_))
        X_train_cnn = X_train_final.reshape((X_train_final.shape[0], X_train_final.shape[1], 1))

        X_val_base = df_completo_scaled.loc[idx_val, features].values
        if usar_if_consejero:
            X_val_final = np.column_stack((X_val_base, df_completo.loc[idx_val, 'prediccion_ia'].values))
        else:
            X_val_final = X_val_base
        X_val_cnn = X_val_final.reshape((X_val_final.shape[0], X_val_final.shape[1], 1))
        y_val_num = le.transform(df_completo.loc[idx_val, 'tipo_ataque'].values)
        y_val_cat = to_categorical(y_val_num, num_classes=len(le.classes_))

        modelo = Sequential([
            Input(shape=(X_train_final.shape[1], 1)),
            Conv1D(filters=32, kernel_size=3, activation='relu'), MaxPooling1D(pool_size=2), Flatten(),
            Dense(64, activation='relu'),
            Dense(len(le.classes_), activation='softmax')
        ])
        modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        modelo.fit(
            X_train_cnn, y_train_cat, epochs=50, batch_size=256, verbose=0,
            validation_data=(X_val_cnn, y_val_cat),
            class_weight=pesos_dict, callbacks=[early_stopping]
        )
    elif tipo_modelo == 'CNN_2D':
        print(f"[AI] Entrenando CNN 2D...")
        y_train_cat = to_categorical(y_train_num, num_classes=len(le.classes_))
        num_features = X_train_final.shape[1]
        D = math.ceil(math.sqrt(num_features))
        pad_size = (D * D) - num_features
        X_train_padded = np.pad(X_train_final, ((0, 0), (0, pad_size)), 'constant')
        X_train_2d = X_train_padded.reshape(-1, D, D, 1)

        X_val_base = df_completo_scaled.loc[idx_val, features].values
        if usar_if_consejero:
            X_val_final = np.column_stack((X_val_base, df_completo.loc[idx_val, 'prediccion_ia'].values))
        else:
            X_val_final = X_val_base
        X_val_padded = np.pad(X_val_final, ((0, 0), (0, pad_size)), 'constant')
        X_val_2d = X_val_padded.reshape(-1, D, D, 1)
        y_val_num = le.transform(df_completo.loc[idx_val, 'tipo_ataque'].values)
        y_val_cat = to_categorical(y_val_num, num_classes=len(le.classes_))

        modelo = Sequential([
            Input(shape=(D, D, 1)),
            Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
            MaxPooling2D(pool_size=(2, 2), padding='same'), Flatten(),
            Dense(64, activation='relu'),
            Dense(len(le.classes_), activation='softmax')
        ])
        modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        modelo.fit(
            X_train_2d, y_train_cat, epochs=50, batch_size=256, verbose=0,
            validation_data=(X_val_2d, y_val_cat),
            class_weight=pesos_dict, callbacks=[early_stopping]
        )

    else:
        print(f"[AI] Entrenando Random Forest...")
        modelo = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
        modelo.fit(X_train_final, y_train_num, sample_weight=pesos_clases)

    # Fin de medición de recursos de Entrenamiento
    tiempo_fin_train = time.time()
    monitor_train.__exit__(None, None, None)

    print("\n--- RENDIMIENTO DEL ENTRENAMIENTO ---")
    print(f"   -> Tiempo total invertido: {tiempo_fin_train - tiempo_inicio_train:.2f} s")
    print(f"   -> RAM pico real:          {monitor_train.pico_ram_mb:.2f} MB")
    print(f"   -> CPU pico real:          {monitor_train.pico_cpu_pct:.2f}%")
    print("-------------------------------------\n")

    # ===========================================================
    # 3. PREDICCIÓN GLOBAL E INFERENCIA PARA GENERAR COLUMNAS
    # ===========================================================
    X_total_base = df_completo_scaled[features].values
    if usar_if_consejero:
        X_total_final = np.column_stack((X_total_base, df_completo['prediccion_ia'].values))
        predicciones_if_eval = df_completo['prediccion_ia'].values
    else:
        X_total_final = X_total_base
        predicciones_if_eval = None

    if tipo_modelo == 'CNN_2D':
        X_total_padded = np.pad(X_total_final, ((0, 0), (0, pad_size)), 'constant')
        X_total_inf = X_total_padded.reshape(-1, D, D, 1)
        probabilidades = modelo.predict(X_total_inf, batch_size=1024, verbose=0)
    elif tipo_modelo in ['ANN', 'CNN']:
        if tipo_modelo == 'CNN':
            X_total_inf = X_total_final.reshape((X_total_final.shape[0], X_total_final.shape[1], 1))
        else:
            X_total_inf = X_total_final
        probabilidades = modelo.predict(X_total_inf, batch_size=1024, verbose=0)
    else:
        probabilidades = modelo.predict_proba(X_total_final)

    clases_modelo = le.classes_ if hasattr(le, 'classes_') else modelo.classes_

    posiciones_val = df_completo.index.get_indexer(idx_val)
    y_real_val = df_completo.loc[idx_val, 'tipo_ataque'].values
    umbrales_ajustados = optimizar_umbrales_por_clase(
        probabilidades[posiciones_val], clases_modelo, y_real_val
    )
    print(f"   -> Umbrales ajustados en validación: {umbrales_ajustados}")

    df_completo['prediccion_ataque'], df_completo['confianza_ataque'] = _aplicar_umbral_por_clase(
        probabilidades, clases_modelo, predicciones_if_eval, umbrales=umbrales_ajustados
    )
    df_completo = filtro_histeresis(df_completo, n_consecutivos=3)

    return modelo_if, modelo, df_completo, features, scaler, umbrales_ajustados

In [49]:
# @title
# ===================================================================
# CELDA 6: EVALUACIÓN CIENTÍFICA + MÉTRICAS DE TEST Y GPU (DEFINITIVA)
# ===================================================================

def obtener_memoria_gpu():
    """Obtiene el consumo actual de VRAM de la GPU en MB si está disponible."""
    try:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            info = tf.config.experimental.get_memory_info('GPU:0')
            return info['current'] / (1024 * 1024)
    except Exception:
        pass
    return 0.0

def evaluar_rendimiento(df_evaluacion, modelo_cl, features, scaler, sufijo_modo, metricas_recursos=None):
    print(f"\n=============================================================")
    print(f" GENERANDO EVALUACIÓN Y REPORTES: {sufijo_modo}")
    print(f"=============================================================")

    carpeta_salida = f"{PATH_DRIVE}/Graficas_TFG/{sufijo_modo}"
    os.makedirs(carpeta_salida, exist_ok=True)

    # FILTRADO ESTRICTO PARA PRODUCCIÓN (FASE TEST Y JUGADOR AMIGO)
    df_test = df_evaluacion[(df_evaluacion['fase'] == 'test') & (df_evaluacion['tipo_conexion'] == 'amigo')].copy()
    df_test = df_test.sort_values(['sesion', 'player_id', 'timestamp_real'])

    if df_test.empty:
        print("[AVISO] No hay datos de test para evaluar.")
        return

    # --- MEDICIÓN DE RECURSOS EXCLUSIVA PARA EL SET DE TEST ---
    vram_antes = obtener_memoria_gpu()
    monitor_test = MonitorRecursos(intervalo=0.2)
    monitor_test.__enter__()

    tiempo_inicio_test = time.time()

    # Estandarizamos los datos reales de Test
    X_test_eval = scaler.transform(df_test[features])

    # Añadimos la columna del consejero si la ablación lo requiere
    if 'Con_Consejero' in sufijo_modo:
        X_test_eval = np.column_stack((X_test_eval, df_test['prediccion_ia'].values))

    # FORZAMOS LA INFERENCIA POR LOTES (rendimiento agregado, no latencia individual)
    if hasattr(modelo_cl, 'predict_proba'):
        _ = modelo_cl.predict_proba(X_test_eval)
    else:
        if '2D' in sufijo_modo:
            num_features = X_test_eval.shape[1]
            D = math.ceil(math.sqrt(num_features))
            pad_size = (D * D) - num_features
            X_test_eval = np.pad(X_test_eval, ((0, 0), (0, pad_size)), 'constant').reshape(-1, D, D, 1)
        elif 'CNN' in sufijo_modo:
            X_test_eval = X_test_eval.reshape((X_test_eval.shape[0], X_test_eval.shape[1], 1))

        _ = modelo_cl.predict(X_test_eval, batch_size=1024, verbose=0)

    tiempo_fin_test = time.time()
    monitor_test.__exit__(None, None, None)

    cpu_usada_test = monitor_test.pico_cpu_pct
    memoria_consumida_test = monitor_test.pico_ram_mb
    vram_consumida = max(0, obtener_memoria_gpu() - vram_antes)
    tiempo_test_total = tiempo_fin_test - tiempo_inicio_test
    latencia_media_lote_ms = (tiempo_test_total / len(df_test)) * 1000

    # --- LATENCIA REAL POR MUESTRA INDIVIDUAL (fila a fila) ---
    NUM_MUESTRAS_LATENCIA = min(200, len(df_test))
    indices_muestra = np.random.RandomState(SEMILLA_GLOBAL).choice(
        len(df_test), size=NUM_MUESTRAS_LATENCIA, replace=False
    )

    tiempos_individuales = []
    for idx in indices_muestra:
        fila = X_test_eval[idx:idx + 1]
        tiempo_inicio_fila = time.time()
        if hasattr(modelo_cl, 'predict_proba'):
            _ = modelo_cl.predict_proba(fila)
        else:
            _ = modelo_cl.predict(fila, verbose=0)
        tiempos_individuales.append((time.time() - tiempo_inicio_fila) * 1000)

    latencia_media_ms = float(np.mean(tiempos_individuales))
    latencia_p95_ms = float(np.percentile(tiempos_individuales, 95))

    ruta_reporte = f"{carpeta_salida}/reporte_metricas_{sufijo_modo}.txt"
    with open(ruta_reporte, 'w', encoding='utf-8') as f:
        f.write(f"REPORTE NIDS - {sufijo_modo}\n")
        try:
            f.write(f"Umbrales: {CONF_THRESHOLDS}\n\n")
        except NameError:
            f.write("Umbrales: No especificados dinámicamente\n\n")

        f.write("=== RENDIMIENTO COMPUTACIONAL (INFERENCIA TEST) ===\n")
        f.write(f"Muestras procesadas:      {len(df_test)}\n")
        f.write(f"Tiempo Total Inferencia:  {tiempo_test_total:.3f} s\n")
        f.write(f"Latencia por lote (batch):        {latencia_media_lote_ms:.4f} ms/muestra\n")
        f.write(f"Latencia real por muestra (p50):  {latencia_media_ms:.4f} ms\n")
        f.write(f"Latencia real por muestra (p95):  {latencia_p95_ms:.4f} ms\n")
        f.write(f"Incremento RAM Peak:      {memoria_consumida_test:.2f} MB\n")
        f.write(f"Incremento VRAM GPU:      {vram_consumida:.2f} MB\n")
        f.write(f"Uso CPU Peak:             {cpu_usada_test}%\n")
        f.write("===================================================\n\n")

        # 1. ISOLATION FOREST
        if 'prediccion_ia' in df_test.columns:
            y_real_bin = df_test['etiqueta_real']
            y_pred_bin = df_test['prediccion_ia']
            reporte_if = classification_report(y_real_bin, y_pred_bin, labels=[-1, 1], zero_division=0)
            f.write("=== 1. ISOLATION FOREST ===\n" + reporte_if + "\n")
            print(f"=== 1. ISOLATION FOREST ===\n{reporte_if}")

            matriz_bin = confusion_matrix(y_real_bin, y_pred_bin, labels=[-1, 1])
            plt.figure(figsize=(6, 4))
            sns.heatmap(matriz_bin, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Ataque (-1)', 'Normal (1)'], yticklabels=['Ataque (-1)', 'Normal (1)'])
            plt.title(f'Matriz IF ({sufijo_modo})')
            plt.savefig(f'{carpeta_salida}/matriz_confusion_if.png', dpi=300, bbox_inches='tight')
            plt.close()

        # 2. CLASIFICADOR MULTICLASE
        y_real_multi = df_test['tipo_ataque']
        clases_posibles = ['ninguno', 'LagSwitch', 'Flood', 'Aimbot', 'Anomalia_Desconocida']
        clases_presentes = [c for c in clases_posibles if c in y_real_multi.unique() or c in df_test['prediccion_ataque_suavizada'].unique()]
        if 'ninguno' not in clases_presentes: clases_presentes.insert(0, 'ninguno')

        for pred_col, titulo in [('prediccion_ataque', 'Modelo por fila'), ('prediccion_ataque_suavizada', 'Modelo temporal suavizado')]:
            f.write(f"=== 2. {titulo.upper()} ===\n")
            reporte = classification_report(y_real_multi, df_test[pred_col], labels=clases_presentes, zero_division=0)
            f.write(reporte + "\n")
            print(f"\n{titulo}:\n{reporte}")

            matriz = confusion_matrix(y_real_multi, df_test[pred_col], labels=clases_presentes)
            plt.figure(figsize=(8, 6))
            sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
                        xticklabels=clases_presentes, yticklabels=clases_presentes)
            plt.title(f'{titulo} ({sufijo_modo})')
            plt.ylabel('Verdad Absoluta')
            plt.xlabel('Predicción NIDS')
            plt.savefig(f'{carpeta_salida}/matriz_confusion_{pred_col}.png', dpi=300, bbox_inches='tight')
            plt.close()

        # 3. DETECCIÓN POR EVENTOS (CONTEO DE TIEMPO Y MUESTRAS POR EVENTO)
        f.write("\n=== 3. DETECCIÓN POR EVENTOS (TIEMPO Y MUESTRAS) ===\n")
        es_ataque_real = df_test['etiqueta_real'] == -1

        cambio_sesion = df_test['sesion'] != df_test['sesion'].shift()
        cambio_estado = es_ataque_real != es_ataque_real.shift()
        id_eventos = (cambio_estado | cambio_sesion).cumsum()

        bloques_ataque = df_test[es_ataque_real].groupby(id_eventos)
        total = len(bloques_ataque)

        cazados_if, cazados_tipo, cazados_cualquier_ataque = 0, 0, 0
        tiempos_deteccion, muestras_deteccion = [], []

        print("\n" + '='*65)
        print("DESGLOSE EXACTO DE TIEMPO Y MUESTRAS POR EVENTO")
        print('='*65)

        for i, (_, bloque) in enumerate(bloques_ataque, 1):
            if bloque.empty: continue

            t_inicio = bloque['timestamp_real'].min()
            tipos_reales = [t for t in bloque['tipo_ataque'].unique() if t != 'ninguno']
            tipo_real = tipos_reales[0] if tipos_reales else "Desconocido"

            # Detección IF
            if 'prediccion_ia' in bloque.columns and (bloque['prediccion_ia'] == -1).any():
                cazados_if += 1
                t_det_if = bloque[bloque['prediccion_ia'] == -1]['timestamp_real'].min()
                tiempo_if = float(t_det_if - t_inicio)
                muestras_if = len(bloque[bloque['timestamp_real'] <= t_det_if])
                msg_if = f"IF lo detectó en {tiempo_if:.3f}s ({muestras_if} muestras)."
            else:
                msg_if = "IF NO lo detectó (o no aplica)."

            # Detección del Clasificador Principal (Suavizada)
            predicciones_bloque = bloque['prediccion_ataque_suavizada']

            # Detectado como ALGO (cualquier ataque)
            if (predicciones_bloque != 'ninguno').any():
                cazados_cualquier_ataque += 1

            # Detectado CORRECTAMENTE (tipo exacto)
            if (predicciones_bloque == tipo_real).any():
                cazados_tipo += 1
                t_det_rf = bloque[predicciones_bloque == tipo_real]['timestamp_real'].min()
                tiempo_rf = float(t_det_rf - t_inicio)
                muestras_rf = len(bloque[bloque['timestamp_real'] <= t_det_rf])

                tiempos_deteccion.append(tiempo_rf)
                muestras_deteccion.append(muestras_rf)
                msg_rf = f"NIDS clasificó {tipo_real} en {tiempo_rf:.3f}s ({muestras_rf} muestras)."
            else:
                msg_rf = f"NIDS falló al clasificar {tipo_real}."

            print(f"Evento {i:02d} | {msg_if} | {msg_rf}")
            f.write(f"Evento {i:02d} | {msg_if} | {msg_rf}\n")

        tasa_if = 100 * cazados_if / total if total else 0
        tasa_cualquiera = 100 * cazados_cualquier_ataque / total if total else 0
        tasa_tipo = 100 * cazados_tipo / total if total else 0
        media_tiempo = np.mean(tiempos_deteccion) if tiempos_deteccion else 0.0
        media_muestras = np.mean(muestras_deteccion) if muestras_deteccion else 0.0

        # Bloque de resumen final unificado (consola + TXT)
        resumen_final = (
            f"-----------------------------------------------------------------\n"
            f"RESUMEN FINAL DE EVENTOS Y RENDIMIENTO:\n"
            f"Eventos de ataque reales:             {total}\n"
            f"Eventos detectados (cualquier tipo):  {cazados_cualquier_ataque} ({tasa_cualquiera:.2f}%)\n"
            f"Eventos correctos (por tipo):         {cazados_tipo} ({tasa_tipo:.2f}%)\n"
            f"Tiempo medio de detección:            {media_tiempo:.3f} s\n"
            f"Muestras medias para detección:       {media_muestras:.1f}\n"
            f"Latencia real por muestra (p50/p95):  {latencia_media_ms:.4f} / {latencia_p95_ms:.4f} ms\n"
            f"Uso VRAM GPU:                         {vram_consumida:.2f} MB\n"
            f"=================================================================\n"
        )

        print(resumen_final)
        f.write("\n" + resumen_final)

    # 4. GRÁFICA TEMPORAL DE BARRAS DE COLOR (HEATMAP) EN 3 FILAS (IMSHOW)
    print(f"\n[VISUALIZACIÓN] Generando líneas de tiempo de barras rojas (3 filas)...")

    cmap_personalizado = ListedColormap(['red', '#1f77b4'])
    col_pred = 'prediccion_ataque_suavizada' if 'prediccion_ataque_suavizada' in df_test.columns else 'prediccion_ataque'
    tiene_if = 'prediccion_ia' in df_test.columns

    for sesion in df_test['sesion'].unique():
        df_sesion = df_test[df_test['sesion'] == sesion].copy().sort_values('timestamp_real')

        tipo_red_actual = df_sesion['tipo_red'].iloc[0] if 'tipo_red' in df_sesion.columns else 'Desconocido'
        ataques_unicos = [a for a in df_sesion['tipo_ataque'].unique() if a != 'ninguno']
        ataque_actual = ataques_unicos[0] if ataques_unicos else 'General'

        real_data = df_sesion['etiqueta_real'].values[np.newaxis, :]
        pred_clasificador = df_sesion[col_pred].values
        clasif_binary = np.where(pred_clasificador != 'ninguno', -1, 1)[np.newaxis, :]

        if tiene_if:
            if_data = df_sesion['prediccion_ia'].values[np.newaxis, :]
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 5.5), sharex=True)

            ax1.imshow(real_data, aspect='auto', cmap=cmap_personalizado, vmin=-1, vmax=1, interpolation='nearest')
            ax1.set_yticks([])
            ax1.set_ylabel('1. Verdad Absoluta\n(Lo que pasó)', rotation=0, labelpad=75, va='center', fontweight='bold', fontsize=10)

            ax2.imshow(if_data, aspect='auto', cmap=cmap_personalizado, vmin=-1, vmax=1, interpolation='nearest')
            ax2.set_yticks([])
            ax2.set_ylabel('2. Detección IF\n(Anomalía NIDS)', rotation=0, labelpad=75, va='center', fontweight='bold', fontsize=10)

            ax3.imshow(clasif_binary, aspect='auto', cmap=cmap_personalizado, vmin=-1, vmax=1, interpolation='nearest')
            ax3.set_yticks([])
            ax3.set_ylabel('3. Clasificador\n(Tipo de Trampa)', rotation=0, labelpad=75, va='center', fontweight='bold', fontsize=10)

            ax3.set_xlabel('Evolución temporal (Muestras/Filas del conjunto de TEST)', fontsize=11)
            plt.subplots_adjust(hspace=0.35)
        else:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 4.0), sharex=True)

            ax1.imshow(real_data, aspect='auto', cmap=cmap_personalizado, vmin=-1, vmax=1, interpolation='nearest')
            ax1.set_yticks([])
            ax1.set_ylabel('1. Verdad Absoluta', rotation=0, labelpad=75, va='center', fontweight='bold', fontsize=10)

            ax2.imshow(clasif_binary, aspect='auto', cmap=cmap_personalizado, vmin=-1, vmax=1, interpolation='nearest')
            ax2.set_yticks([])
            ax2.set_ylabel('2. Clasificador\n(Tipo de Trampa)', rotation=0, labelpad=75, va='center', fontweight='bold', fontsize=10)

            ax2.set_xlabel('Evolución temporal (Muestras/Filas del conjunto de TEST)', fontsize=11)
            plt.subplots_adjust(hspace=0.30)

        sesion_nombre_corto = os.path.basename(str(sesion))[:15]
        fig.suptitle(f'Línea de Tiempo: Verdad Absoluta vs Detección NIDS\nTrampa: {str(ataque_actual).upper()} | Red: {str(tipo_red_actual).upper()} | Sesión: {sesion_nombre_corto}',
                     fontsize=13, fontweight='bold', y=1.02)

        leyenda = [
            Patch(color='#1f77b4', label='Tráfico Normal (Limpio)'),
            Patch(color='red', label='Anomalía / Ataque (Trampa)')
        ]
        fig.legend(handles=leyenda, loc='upper right', bbox_to_anchor=(0.98, 1.02))

        plt.subplots_adjust(hspace=0.35)

        ataque_limpio = str(ataque_actual).replace('/', '_').replace(' ', '_')
        sesion_limpia = str(sesion_nombre_corto).replace('/', '_').replace(' ', '_')

        ruta_grafica = f"{carpeta_salida}/timeline_barras_{ataque_limpio}_{sesion_limpia}.png"
        plt.savefig(ruta_grafica, dpi=300, bbox_inches='tight')
        plt.close()

    print(f"[OK] Gráficas de barras en 3 filas guardadas en: {carpeta_salida}")
    print(f"[OK] Reporte TXT completo guardado en: {ruta_reporte}")

In [50]:
# @title
# ===================================================================
# CELDA 7: CONFIGURACIÓN DEL EXPERIMENTO
# ===================================================================
ARCHIVO_SYNC_GLOBAL = f"{PATH_DRIVE_DATA}/sincronizacion_tiempos.txt"

# VARIABLE DE CONTROL GLOBAL PARA LA COMPARATIVA
# - False: Ejecuta el modelo básico (sin ventanas ni velocidad angular)
# - True: Ejecuta el modelo avanzado con ventanas deslizantes y memoria
USAR_VENTANAS_Y_MEMORIA = True

sesiones_a_procesar = [
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/dataset_telemetria_20260702_215841.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": False, "fase": "train"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/dataset_telemetria_20260702_220345.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": False, "fase": "train"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/dataset_telemetria_20260702_220826.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": False, "fase": "train"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/dataset_telemetria_20260702_221150.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": False, "fase": "train"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas27072026LagSwitchRandom/dataset_telemetria_20260727_180057.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas27072026LagSwitchRandom/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas27072026LagSwitchRandom/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "LagSwitch", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas17082026LagSwitchRandom/dataset_telemetria_20260817_155143.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas17082026LagSwitchRandom/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas17082026LagSwitchRandom/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "LagSwitch", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas27072026FloodRandom/dataset_telemetria_20260727_175216.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas27072026FloodRandom/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas27072026FloodRandom/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Flood", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas17082026FloodRandom/dataset_telemetria_20260817_161917.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas17082026FloodRandom/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas17082026FloodRandom/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Flood", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/1/dataset_telemetria_20260817_164738.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/1/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/1/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Aimbot", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/2/dataset_telemetria_20260817_165327.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/2/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas17092026AimbotRandom/2/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Aimbot", "fase": "train_rf"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas12072026LagSwitch/dataset_telemetria_20260712_175633.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas12072026LagSwitch/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas12072026LagSwitch/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "LagSwitch", "fase": "test"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas27072026Flood/dataset_telemetria_20260727_164251.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas27072026Flood/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas27072026Flood/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Flood", "fase": "test"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/dataset_telemetria_20260702_221530.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas02072026Bueno/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": False, "fase": "test"
    },
    {
        "csv": f"{PATH_DRIVE_DATA}/Partidas17082026AimbotTest/dataset_telemetria_20260817_163858.csv",
        "pcap_local": f"{PATH_DRIVE_DATA}/Partidas17082026AimbotTest/trafico_local.pcap",
        "pcap_amigo": f"{PATH_DRIVE_DATA}/Partidas17082026AimbotTest/trafico_amigo.pcap",
          "tipo_red": "5G", "es_ataque": True, "tipo_ataque": "Aimbot", "fase": "test"
    }
]

def extraer_fecha_de_csv(ruta_csv):
    match = re.search(r"(\d{8}_\d{6})", ruta_csv)
    if not match:
        raise ValueError(f"No se pudo extraer fecha de {ruta_csv}")
    return match.group(1)

sesiones_aimbot_ordenadas = sorted(
    [s for s in sesiones_a_procesar if s.get('tipo_ataque') == 'Aimbot'],
    key=lambda s: extraer_fecha_de_csv(s['csv'])
)

for indice_bloque, sesion in enumerate(sesiones_aimbot_ordenadas):
    sesion['bloque_aimbot'] = indice_bloque

In [51]:
# @title
# ===================================================================
# CELDA 8: EJECUCIÓN OPTIMIZADA DE EXPERIMENTOS (ESTUDIO DE ABLACIÓN)
# ===================================================================
import gc
import os
import joblib
import pandas as pd
import numpy as np
import sys

# 1. Definir los tres niveles base que se procesarán
NIVELES_DATASET = [
    # ("Dataset_Basico", False, False),
    # ("Dataset_Avanzado", True, False),
    ("Dataset_V3_Multiescala", True, True)
]

# 2. Definir los modelos que se evaluarán sobre cada dataset
MODELOS_A_PROBAR = [
    ('RF_Con_Consejero', 'RF', True),
    ('RF_Sin_Consejero', 'RF', False),
    ('XGB_Con_Consejero', 'XGB', True),
    ('XGB_Sin_Consejero', 'XGB', False),
    ('ANN', 'ANN', False),
    ('CNN', 'CNN', False), # Ideal para V3
    ('CNN_2D', 'CNN_2D', False)
]

# Creamos la carpeta específica para alojar los datasets base
carpeta_datasets = f"{PATH_DRIVE}/Datasets_TFG"
os.makedirs(carpeta_datasets, exist_ok=True)

for nombre_nivel, modo_actual, modo_v3 in NIVELES_DATASET:
    print(f"\n\n{'='*70}\n GENERANDO NIVEL DE DATOS: {nombre_nivel}\n{'='*70}")

    ruta_dataset_base = f"{carpeta_datasets}/{nombre_nivel}_BASE.csv"
    ruta_reporte_generacion = f"{carpeta_datasets}/reporte_generacion_{nombre_nivel}.txt"

    ruta_manifiesto_dataset = f"{carpeta_datasets}/manifiesto_config_{nombre_nivel}.pkl"
    configuracion_actual = {
        'usar_ventanas': modo_actual,
        'usar_v3': modo_v3,
        'sesiones': [
            (s['csv'], s['fase'], s.get('tipo_ataque', 'ninguno'))
            for s in sesiones_a_procesar
        ]
    }

    # Verificamos si ya existe el dataset base guardado para reutilizarlo
    if os.path.exists(ruta_dataset_base):
        print(f"[AI] Reutilizando dataset base existente desde: {ruta_dataset_base}")
        dataset_maestro = pd.read_csv(ruta_dataset_base)

        if os.path.exists(ruta_manifiesto_dataset):
            configuracion_guardada = joblib.load(ruta_manifiesto_dataset)
            if configuracion_guardada != configuracion_actual:
                raise ValueError(
                    f"El dataset '{ruta_dataset_base}' se generó con otra configuración "
                    f"(usar_v3, usar_ventanas o lista de sesiones distinta). "
                    f"Borra el CSV para regenerarlo o revisa 'sesiones_a_procesar'."
                )
        else:
            print(
                f"[AVISO] No hay manifiesto de configuración para '{ruta_dataset_base}'; "
                f"no se puede verificar que coincida con la configuración actual."
            )

        print(f"[OK] Dataset {nombre_nivel} cargado correctamente. Filas totales: {len(dataset_maestro)}")
    else:
        # Clase auxiliar para redirigir la salida por consola al archivo de reporte de primera ejecución
        class Tee(object):
            def __init__(self, filename):
                self.terminal = sys.stdout
                self.log = open(filename, "w", encoding="utf-8")
            def write(self, message):
                self.terminal.write(message)
                self.log.write(message)
                self.log.flush()
            def flush(self):
                self.terminal.flush()
                self.log.flush()
            def close(self):
                if not self.log.closed:
                    self.log.close()

        # Guardamos temporalmente el stdout original
        stdout_original = sys.stdout
        tee_actual = Tee(ruta_reporte_generacion)
        sys.stdout = tee_actual

        try:
            rutas_temporales = []

            for i, sesion in enumerate(sesiones_a_procesar, 1):
                print(f"--- Procesando sesión {i}/{len(sesiones_a_procesar)} para {nombre_nivel} ---")
                desfase = obtener_desfase_por_csv(ARCHIVO_SYNC_GLOBAL, sesion['csv'])

                # Procesamiento pesado de la sesión
                df_sesion = preprocesar_y_fusionar(
                    sesion['csv'], sesion['pcap_local'], sesion['pcap_amigo'], desfase,
                    sesion['tipo_red'], sesion['es_ataque'], sesion['fase'],
                    sesion.get('tipo_ataque', 'ninguno'), usar_ventanas=modo_actual, usar_v3=modo_v3,
                    bloque_aimbot=sesion.get('bloque_aimbot')
                )

                if df_sesion is None or df_sesion.empty:
                    continue

                # --- REPORTE DE ESTADÍSTICAS POR PARTIDA (Filtrado solo para el amigo) ---
                df_sesion_amigo = df_sesion[df_sesion['tipo_conexion'] == 'amigo']

                filas_totales = len(df_sesion_amigo)
                filas_buenas = len(df_sesion_amigo[df_sesion_amigo['etiqueta_real'] == 1])
                filas_malas = len(df_sesion_amigo[df_sesion_amigo['etiqueta_real'] == -1])

                # Conteo de eventos reales en esta sesión para el amigo
                es_ataque_real = df_sesion_amigo['etiqueta_real'] == -1
                id_eventos = (es_ataque_real != es_ataque_real.shift()).cumsum()
                num_eventos = len(df_sesion_amigo[es_ataque_real].groupby(id_eventos))

                print(f"      [ESTADÍSTICAS] Filas Totales: {filas_totales} | Buenas: {filas_buenas} | Ataques (Malas): {filas_malas}")
                print(f"      [ESTADÍSTICAS] Eventos de trampa inyectados: {num_eventos}")

                cols_float = df_sesion.select_dtypes(include=['float64']).columns
                cols_float = [c for c in cols_float if 'timestamp' not in c]
                df_sesion[cols_float] = df_sesion[cols_float].astype('float32')

                cols_obj = df_sesion.select_dtypes(include=['object']).columns
                df_sesion[cols_obj] = df_sesion[cols_obj].astype(str)

                ruta_temp = f"/content/temp_sesion_{i}.pkl"
                df_sesion.to_pickle(ruta_temp)
                rutas_temporales.append(ruta_temp)

                del df_sesion
                gc.collect()

            print(f"\n[AI] Uniendo dataset base {nombre_nivel} desde el disco duro...")
            dataset_maestro = pd.concat([pd.read_pickle(ruta) for ruta in rutas_temporales], ignore_index=True)

            dataset_maestro = asignar_validacion_por_eventos(dataset_maestro)

            for ruta in rutas_temporales:
                if os.path.exists(ruta): os.remove(ruta)
            gc.collect()

            dataset_maestro.to_csv(ruta_dataset_base, index=False)
            joblib.dump(configuracion_actual, ruta_manifiesto_dataset)
            print(f"[OK] Dataset {nombre_nivel} ensamblado. Filas totales: {len(dataset_maestro)}")

        finally:
            # Restauramos el stdout original y cerramos el archivo de reporte
            sys.stdout = stdout_original
            tee_actual.close()
            print(f"[OK] Reporte de la primera ejecución guardado en: {ruta_reporte_generacion}")

    ruta_informe_calidad = f"{carpeta_datasets}/informe_calidad_{nombre_nivel}.txt"
    generar_informe_calidad_dataset(dataset_maestro, ruta_salida=ruta_informe_calidad)

    # --- ENTRENAMIENTO DE MODELOS SOBRE ESTE DATASET BASE ---
    for sufijo_modelo, tipo_modelo, usa_if in MODELOS_A_PROBAR:

        sufijo_completo = f"{nombre_nivel}_{sufijo_modelo}"
        print(f"\n\n{'='*50}\n ENTRENANDO EXPERIMENTO: {sufijo_completo}\n{'='*50}")

        modelo_if, modelo_cl, dataset_final, features, scaler, umbrales_ajustados = entrenar_nids(
            dataset_maestro.copy(), usar_ventanas=modo_actual, usar_v3=modo_v3,
            tipo_modelo=tipo_modelo, usar_if_consejero=usa_if
        )

        ruta_modelos = f"{PATH_DRIVE}/Modelos_TFG/{sufijo_completo}"
        os.makedirs(ruta_modelos, exist_ok=True)

        if modelo_if is not None:
            joblib.dump(modelo_if, f'{ruta_modelos}/modelo_if.pkl')

        joblib.dump(scaler, f'{ruta_modelos}/scaler.pkl')
        joblib.dump({'features': features, 'thresholds': umbrales_ajustados, 'if_contamination': IF_CONTAMINATION}, f'{ruta_modelos}/config.pkl')
        hash_dataset = hashlib.sha256(
            pd.util.hash_pandas_object(dataset_maestro, index=True).values
        ).hexdigest()

        manifiesto = {
            'fecha_entrenamiento_utc': datetime.now(timezone.utc).isoformat(),
            'semilla_global': SEMILLA_GLOBAL,
            'tipo_modelo': tipo_modelo,
            'usa_if_consejero': usa_if,
            'nombre_nivel_dataset': nombre_nivel,
            'hash_dataset_maestro': hash_dataset,
            'num_filas_dataset': len(dataset_maestro),
            'versiones': {
                'python': sys.version,
                'sistema_operativo': platform.platform(),
                'pandas': pd.__version__,
                'numpy': np.__version__,
                'scikit_learn': sklearn.__version__,
                'xgboost': xgb.__version__,
                'tensorflow': tf.__version__,
            }
        }

        joblib.dump(manifiesto, f'{ruta_modelos}/manifiesto.pkl')

        if tipo_modelo in ['ANN', 'CNN', 'CNN_2D']:
            modelo_cl.save(f'{ruta_modelos}/modelo_keras.keras')
        else:
            joblib.dump(modelo_cl, f'{ruta_modelos}/modelo_ml.pkl')

        evaluar_rendimiento(dataset_final, modelo_cl, features, scaler, sufijo_modo=sufijo_completo)
        del dataset_final
        gc.collect()

    del dataset_maestro
    gc.collect()

print("\n\n TODOS LOS EXPERIMENTOS Y NIVELES HAN FINALIZADO.")



 GENERANDO NIVEL DE DATOS: Dataset_V3_Multiescala
--- Procesando sesión 1/14 para Dataset_V3_Multiescala ---
   -> Buscando sincronización para /content/drive/MyDrive/NIDS/Partidas/Partidas02072026Bueno/dataset_telemetria_20260702_215841.csv...
      [OK] Sincronización encontrada. Desfase: 1783029518.522 s
   -> Telemetría leída: 33249 filas.
      [AUTO-ID] Amigo=10 | Host=9
      [RED] Filas sin correspondencia directa en el PCAP: 50.09%
      [ESTADÍSTICAS] Filas Totales: 7884 | Buenas: 7884 | Ataques (Malas): 0
      [ESTADÍSTICAS] Eventos de trampa inyectados: 0
--- Procesando sesión 2/14 para Dataset_V3_Multiescala ---
   -> Buscando sincronización para /content/drive/MyDrive/NIDS/Partidas/Partidas02072026Bueno/dataset_telemetria_20260702_220345.csv...
      [OK] Sincronización encontrada. Desfase: 1783029824.065 s
   -> Telemetría leída: 38573 filas.
      [AUTO-ID] Amigo=9 | Host=10
      [RED] Filas sin correspondencia directa en el PCAP: 50.18%
      [ESTADÍSTICAS] Filas T

In [ ]:
# @title
# ===================================================================
# CELDA 9: VALIDACION DIFERIDA VS NIDS EN VIVO
# ===================================================================

PATH_DRIVE_DATA = PATH_DRIVE + '/tiempo_real'
ARCHIVO_SYNC_GLOBAL = f"{PATH_DRIVE_DATA}/sincronizacion_tiempos.txt"

PARTIDAS_A_COMPARAR = [
    {
        'nombre': 'partida_1',
        'csv_telemetria': f'{PATH_DRIVE_DATA}/aimbot/dataset_telemetria_20260908_170947.csv',
        'pcap_local': f'{PATH_DRIVE_DATA}/aimbot/trafico_local.pcap',
        'pcap_amigo': f'{PATH_DRIVE_DATA}/aimbot/trafico_amigo.pcap',
        'tipo_red': '5G',
        'es_ataque': True,
        'tipo_ataque': 'Aimbot',
        'bloque_aimbot': 0,
        'live_csv': f'{PATH_DRIVE_DATA}/aimbot/nids_stream_live.csv',
    }
]

RUTA_MODELO_COMPARAR = f'{PATH_DRIVE}/Modelos_TFG/Dataset_V3_Multiescala_XGB_Sin_Consejero'
RUTA_SALIDA_COMPARACION = f'{PATH_DRIVE}/Graficas_TFG/comparacion_diferido_vs_live'
TOLERANCIA_ALINEACION_SEC = 0.6


def _cargar_artefactos_nids(ruta_modelo):
    config = joblib.load(os.path.join(ruta_modelo, 'config.pkl'))
    manifiesto = joblib.load(os.path.join(ruta_modelo, 'manifiesto.pkl'))
    scaler = joblib.load(os.path.join(ruta_modelo, 'scaler.pkl'))
    tipo_modelo = manifiesto.get('tipo_modelo', 'RF')
    usa_if = manifiesto.get('usa_if_consejero', False)

    modelo_if = None
    if usa_if:
        ruta_if = os.path.join(ruta_modelo, 'modelo_if.pkl')
        if not os.path.exists(ruta_if):
            raise FileNotFoundError(f'Falta el Isolation Forest: {ruta_if}')
        modelo_if = joblib.load(ruta_if)

    if tipo_modelo in ['ANN', 'CNN', 'CNN_2D']:
        from tensorflow.keras.models import load_model
        modelo = load_model(os.path.join(ruta_modelo, 'modelo_keras.keras'))
    else:
        modelo = joblib.load(os.path.join(ruta_modelo, 'modelo_ml.pkl'))

    return config, manifiesto, scaler, modelo_if, modelo, tipo_modelo


def _preparar_caracteristicas(df, features):
    faltantes = [feature for feature in features if feature not in df.columns]
    if faltantes:
        raise ValueError(
            'El preprocesado no genero las features del modelo: '
            + ', '.join(faltantes)
        )
    return (
        df[features]
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
    )


def _predecir_con_artefactos(df, artefactos):
    config, manifiesto, scaler, modelo_if, modelo, tipo_modelo = artefactos
    features = config['features']
    x_base = scaler.transform(_preparar_caracteristicas(df, features))

    pred_if = None
    if modelo_if is not None:
        candidatas_if = [
            'bidirectional_stddev_piat_ms',
            'src2dst_packets_sum',
            'bidirectional_bytes_sum',
            'delta_yaw',
            'vel_angular_yaw',
            'src2dst_packets_sum_roll_max_5',
            'bidirectional_bytes_sum_roll_max_5',
            'bidirectional_mean_piat_ms_roll_std_5',
        ]
        features_if = [feature for feature in candidatas_if if feature in features]
        n_features_if = getattr(modelo_if, 'n_features_in_', len(features_if))
        if len(features_if) != n_features_if:
            raise ValueError(
                f'El IF espera {n_features_if} features y se encontraron '
                f'{len(features_if)}: {features_if}'
            )
        indices_if = [features.index(feature) for feature in features_if]
        pred_if = modelo_if.predict(x_base[:, indices_if])
        x_final = np.column_stack((x_base, pred_if))
    else:
        x_final = x_base

    clases_orden = ['Aimbot', 'Flood', 'LagSwitch', 'ninguno']
    umbrales = config.get('thresholds', {})

    if tipo_modelo == 'CNN_2D':
        dimension = math.ceil(math.sqrt(x_final.shape[1]))
        relleno = dimension * dimension - x_final.shape[1]
        x_inferencia = np.pad(x_final, ((0, 0), (0, relleno)), 'constant')
        probabilidades = modelo.predict(
            x_inferencia.reshape(-1, dimension, dimension, 1), verbose=0
        )
        clases_modelo = np.arange(probabilidades.shape[1])
    elif tipo_modelo in ['ANN', 'CNN']:
        if tipo_modelo == 'CNN':
            x_inferencia = x_final.reshape(x_final.shape[0], x_final.shape[1], 1)
        else:
            x_inferencia = x_final
        probabilidades = modelo.predict(x_inferencia, verbose=0)
        clases_modelo = np.arange(probabilidades.shape[1])
    elif hasattr(modelo, 'predict_proba'):
        probabilidades = modelo.predict_proba(x_final)
        clases_modelo = getattr(modelo, 'classes_', np.arange(probabilidades.shape[1]))
    else:
        predicciones = modelo.predict(x_final)
        predicciones = [
            clases_orden[int(prediccion)] if isinstance(prediccion, (int, np.integer)) else str(prediccion)
            for prediccion in predicciones
        ]
        return np.asarray(predicciones, dtype=object), np.ones(len(predicciones)), pred_if

    indices = np.argmax(probabilidades, axis=1)
    confianza = np.max(probabilidades, axis=1)
    resultado = []
    for posicion, indice in enumerate(indices):
        clase = clases_modelo[indice]
        if isinstance(clase, (int, np.integer)):
            clase = clases_orden[int(clase)]
        else:
            clase = str(clase)
        if clase != 'ninguno' and confianza[posicion] < umbrales.get(clase, 0.0):
            clase = 'ninguno'
        resultado.append(clase)
    return np.asarray(resultado, dtype=object), confianza, pred_if


def _cargar_predicciones_live(ruta_live):
    if not os.path.exists(ruta_live):
        raise FileNotFoundError(f'No existe el CSV de nids_live: {ruta_live}')
    live = pd.read_csv(ruta_live)
    if live.empty:
        raise ValueError(f'El CSV de nids_live esta vacio: {ruta_live}')
    if 'timestamp' not in live or 'player_id' not in live:
        raise ValueError('El CSV live necesita las columnas timestamp y player_id')
    columna_prediccion = 'tipo_ataque' if 'tipo_ataque' in live else 'pred_live'
    if columna_prediccion not in live:
        raise ValueError('El CSV live necesita tipo_ataque o pred_live')
    live = live[['timestamp', 'player_id', columna_prediccion]].copy()
    live = live.rename(columns={columna_prediccion: 'prediccion_live'})
    live['timestamp'] = pd.to_numeric(live['timestamp'], errors='coerce')
    live['player_id'] = pd.to_numeric(live['player_id'], errors='coerce')
    live = live.dropna(subset=['timestamp', 'player_id'])
    live['player_id'] = live['player_id'].astype(int)
    # El daemon puede escribir varias veces el mismo instante; conserva la ultima fila.
    return live.drop_duplicates(['player_id', 'timestamp'], keep='last')


def _alinear_resultados(df_offline, df_live):
    offline = df_offline.sort_values(['timestamp_real', 'player_id']).copy()
    live = df_live.sort_values(['timestamp', 'player_id']).copy()
    return pd.merge_asof(
        offline,
        live,
        left_on='timestamp_real',
        right_on='timestamp',
        by='player_id',
        direction='nearest',
        tolerance=TOLERANCIA_ALINEACION_SEC,
    )


def _imprimir_metricas(y_real, y_pred, titulo):
    etiquetas = ['ninguno', 'Aimbot', 'Flood', 'LagSwitch']
    presentes = [etiqueta for etiqueta in etiquetas if etiqueta in set(y_real) | set(y_pred)]
    print(f'\n=== {titulo} ===')
    print(classification_report(y_real, y_pred, labels=presentes, zero_division=0))
    return presentes


def comparar_diferido_con_live(partidas, ruta_modelo, ruta_salida):
    os.makedirs(ruta_salida, exist_ok=True)
    artefactos = _cargar_artefactos_nids(ruta_modelo)
    config, manifiesto = artefactos[0], artefactos[1]
    features = config['features']
    usar_ventanas = any(feature in features for feature in [
        'vel_angular_yaw', 'vel_angular_pitch', 'acc_angular_yaw',
        'acc_angular_pitch', 'jerk_angular_yaw', 'jerk_angular_pitch'
    ])
    usar_v3 = any('_roll_max_' in feature or '_cv_' in feature for feature in features)
    print(
        f"[VALIDACION] Modelo={manifiesto.get('tipo_modelo', 'RF')} | "
        f"features={len(features)} | ventanas={usar_ventanas} | v3={usar_v3}"
    )

    resultados = []
    for partida in partidas:
        nombre = partida['nombre']
        print(f'\n[PARTIDA] Procesando diferido: {nombre}')
        df_offline = preprocesar_y_fusionar(
            partida['csv_telemetria'],
            partida['pcap_local'],
            partida['pcap_amigo'],
            obtener_desfase_por_csv(ARCHIVO_SYNC_GLOBAL, partida['csv_telemetria']),
            partida.get('tipo_red', 'desconocida'),
            partida.get('es_ataque', False),
            'test',
            partida.get('tipo_ataque', 'ninguno'),
            usar_ventanas=usar_ventanas,
            usar_v3=usar_v3,
            bloque_aimbot=partida.get('bloque_aimbot'),
        )
        df_offline = df_offline[df_offline['tipo_conexion'] == 'amigo'].copy()
        if df_offline.empty:
            raise ValueError(f'No quedaron filas del jugador amigo en {nombre}')

        pred_offline, confianza, _ = _predecir_con_artefactos(df_offline, artefactos)
        df_offline['prediccion_diferida'] = pred_offline
        df_offline['confianza_diferida'] = confianza
        df_offline['partida'] = nombre

        live = _cargar_predicciones_live(partida['live_csv'])
        minimo = df_offline['timestamp_real'].min() - TOLERANCIA_ALINEACION_SEC
        maximo = df_offline['timestamp_real'].max() + TOLERANCIA_ALINEACION_SEC
        live = live[live['timestamp'].between(minimo, maximo)]
        combinado = _alinear_resultados(df_offline, live)
        combinado['partida'] = nombre
        combinado['coincide_live'] = combinado['prediccion_diferida'] == combinado['prediccion_live']
        combinado = combinado.dropna(subset=['prediccion_live']).copy()
        if combinado.empty:
            raise ValueError(
                f'No se pudo alinear {nombre} con {partida["live_csv"]}. '
                f'Revisa timestamps, player_id y la tolerancia de {TOLERANCIA_ALINEACION_SEC}s.'
            )

        etiquetas = _imprimir_metricas(
            combinado['tipo_ataque'], combinado['prediccion_diferida'],
            f'{nombre}: modelo diferido contra ground truth'
        )
        _imprimir_metricas(
            combinado['tipo_ataque'], combinado['prediccion_live'],
            f'{nombre}: nids_live contra ground truth'
        )
        print(
            f'[ALINEACION] {nombre}: {len(combinado)}/{len(df_offline)} filas '
            f'({100 * len(combinado) / len(df_offline):.2f}%) | '
            f'coincidencia diferido/live={100 * combinado["coincide_live"].mean():.2f}%'
        )

        nombre_csv = os.path.join(ruta_salida, f'{nombre}_detalle_comparacion.csv')
        combinado.to_csv(nombre_csv, index=False)
        matriz = confusion_matrix(
            combinado['tipo_ataque'], combinado['prediccion_diferida'], labels=etiquetas
        )
        plt.figure(figsize=(8, 6))
        sns.heatmap(matriz, annot=True, fmt='d', cmap='Oranges',
                    xticklabels=etiquetas, yticklabels=etiquetas)
        plt.title(f'{nombre}: modelo diferido vs ground truth')
        plt.ylabel('Ground truth')
        plt.xlabel('Prediccion diferida')
        plt.tight_layout()
        plt.savefig(os.path.join(ruta_salida, f'{nombre}_matriz_diferido.png'), dpi=200)
        plt.close()
        resultados.append(combinado)

    resultado_total = pd.concat(resultados, ignore_index=True)
    resultado_total.to_csv(os.path.join(ruta_salida, 'comparacion_total.csv'), index=False)
    _imprimir_metricas(
        resultado_total['tipo_ataque'], resultado_total['prediccion_diferida'],
        'TOTAL: modelo diferido contra ground truth'
    )
    _imprimir_metricas(
        resultado_total['tipo_ataque'], resultado_total['prediccion_live'],
        'TOTAL: nids_live contra ground truth'
    )
    print(
        f'\n[OK] Comparacion terminada. Archivos guardados en: {ruta_salida}\n'
        f'[OK] Filas alineadas totales: {len(resultado_total)} | '
        f'Coincidencia diferido/live: {100 * resultado_total["coincide_live"].mean():.2f}%'
    )
    return resultado_total


# Ejecuta la comparacion despues de revisar las rutas de PARTIDAS_A_COMPARAR.
df_comparacion_live = comparar_diferido_con_live(
    PARTIDAS_A_COMPARAR,
    RUTA_MODELO_COMPARAR,
    RUTA_SALIDA_COMPARACION,
)